# 🤖 Introduction to Agents

**Day 10 — Notebook 3 of 5 | Estimated Time: 30 minutes**

---

## What You'll Learn
- What makes something an "agent" vs a standard LLM call
- The perceive → reason → act loop
- Agent types: tool-use, planning, and self-reflective agents
- The ReAct (Reason + Act) pattern
- Memory: in-context, external, and episodic
- When to use agents vs simpler approaches

---

## What Is an Agent?

```
Standard LLM call:          Agent:
────────────────────        ──────────────────────────────────
Input → LLM → Output        Input
                              │
One shot. Done.               ▼
                            Perceive (observe environment)
                              │
                              ▼
                            Reason (plan next action)
                              │
                              ▼
                            Act (call tool or respond)
                              │
                              └─ loop until task complete
```

An agent is an LLM that can **take actions** that affect its environment  
and **iterate** until it achieves a goal.

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os, json, datetime

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from pydantic import BaseModel

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")
MODEL = "gemini-2.5-flash"

---

## 1. Agent Types

Not all agents are the same. Understanding the types helps you pick the right architecture.

In [ ]:
print("""
AGENT TYPES

1. TOOL-USE AGENT (simplest)
   • Receives a task and a toolbox
   • Calls tools as needed to gather information
   • Returns a final answer once it has enough information
   • No explicit planning step
   • Example: the agent from Notebooks 1-2

2. REACT AGENT (Reason + Act)
   • At each step, explicitly THINKS before acting
   • Pattern: Thought → Action → Observation → Thought → ...
   • Scratchpad reasoning is visible and inspectable
   • Good for debugging and interpretability
   • Example: what we build in Notebook 4

3. PLANNING AGENT
   • Decomposes the task into a plan BEFORE taking any actions
   • Executes steps of the plan in order
   • Can revise the plan based on observations
   • Good for long-horizon tasks
   • Example: a research agent that outlines a report first

4. MULTI-AGENT SYSTEM
   • Multiple specialised agents collaborate
   • Orchestrator agent delegates to worker agents
   • Each worker is expert in one domain
   • Example: Notebook 5 — Retriever + Generator + Critic

5. SELF-REFLECTIVE AGENT
   • After producing an answer, agent critiques its own output
   • If quality is low, it tries again with a different approach
   • Improves quality at the cost of additional LLM calls
""")

---

## 2. The ReAct Pattern

ReAct (Reason + Act) adds explicit thinking traces to the agent loop.  
The model is prompted to show its reasoning before each action.

In [ ]:
# Environment for the ReAct demo — a simple world the agent can query
KNOWLEDGE_BASE = {
    "london_population": 9_000_000,
    "paris_population": 2_161_000,
    "tokyo_population": 13_960_000,
    "london_area_km2": 1572,
    "paris_area_km2": 105,
    "tokyo_area_km2": 2194,
    "london_mayor": "Sadiq Khan",
    "paris_mayor": "Anne Hidalgo",
    "tokyo_governor": "Yuriko Koike",
}


def lookup(key: str) -> dict:
    """Look up a fact from the knowledge base."""
    if key in KNOWLEDGE_BASE:
        return {"key": key, "value": KNOWLEDGE_BASE[key]}
    # Try partial match
    matches = [k for k in KNOWLEDGE_BASE if key.lower() in k.lower()]
    if matches:
        return {"available_keys": matches, "hint": f"Try one of: {matches}"}
    return {"error": f"Key '{key}' not found"}


def calculate(expression: str) -> dict:
    """Safely evaluate a simple arithmetic expression."""
    try:
        # Only allow safe operations
        allowed = set("0123456789.+-*/() ")
        if not all(c in allowed for c in expression):
            return {"error": "Only arithmetic expressions allowed"}
        result = eval(expression)  # noqa — safe because of the allowlist above
        return {"expression": expression, "result": round(result, 4)}
    except Exception as e:
        return {"error": str(e)}


REACT_TOOLS = [
    types.Tool(function_declarations=[
        types.FunctionDeclaration(
            name="lookup",
            description="Look up a specific fact. Available keys include: city_population, city_area_km2, city_mayor/governor. Example key: 'london_population'",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={"key": types.Schema(type=types.Type.STRING, description="The fact key to look up")},
                required=["key"],
            ),
        ),
        types.FunctionDeclaration(
            name="calculate",
            description="Evaluate an arithmetic expression. Use standard Python operators: +, -, *, /. Example: '9000000 / 1572'",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={"expression": types.Schema(type=types.Type.STRING, description="Arithmetic expression to evaluate")},
                required=["expression"],
            ),
        ),
    ])
]

REACT_REGISTRY = {"lookup": lookup, "calculate": calculate}

print("ReAct environment ready.")
print(f"Available facts: {list(KNOWLEDGE_BASE.keys())}")

In [ ]:
REACT_SYSTEM_PROMPT = """You are a reasoning agent. For each task:
1. Think step-by-step about what information you need
2. Use tools to gather that information
3. Compute or derive the final answer
4. State your answer clearly at the end

Think before each action. Be precise with calculations."""


def run_react_agent(query: str, max_steps: int = 8) -> str:
    """ReAct agent with visible reasoning traces."""
    contents = [types.Content(
        role="user",
        parts=[types.Part(text=f"{REACT_SYSTEM_PROMPT}\n\nTask: {query}")]
    )]

    step = 0
    print(f"Task: {query}\n")

    while step < max_steps:
        step += 1
        response = client.models.generate_content(
            model=MODEL, contents=contents,
            config=types.GenerateContentConfig(tools=REACT_TOOLS),
        )
        part = response.candidates[0].content.parts[0]

        # Text response — could be a thought or the final answer
        if not hasattr(part, "function_call") or part.function_call is None:
            print(f"[Final Answer]\n{part.text.strip()}")
            return part.text.strip()

        fn_call = part.function_call
        fn_name, fn_args = fn_call.name, dict(fn_call.args)

        print(f"[Step {step}] Action: {fn_name}({fn_args})")

        observation = REACT_REGISTRY[fn_name](**fn_args)
        print(f"           Observation: {observation}")

        contents.append(types.Content(role="model", parts=[types.Part(function_call=fn_call)]))
        contents.append(types.Content(role="user", parts=[types.Part(
            function_response=types.FunctionResponse(name=fn_name, response=observation)
        )]))

    return "Max steps reached."


# Test a multi-step reasoning task
answer = run_react_agent(
    "Which city is more densely populated: London or Paris? "
    "Calculate the density for each (people per km²) and compare."
)

---

## 3. Agent Memory

An agent without memory can't build on prior interactions. There are three kinds of memory:

In [ ]:
print("""
AGENT MEMORY TYPES

1. IN-CONTEXT MEMORY (short-term)
   • The conversation history in the current context window
   • Limited by token window (~1M tokens for Gemini)
   • Lost when the conversation ends
   • Perfect for: multi-turn tasks, tool call history

2. EXTERNAL MEMORY (long-term)
   • Facts stored in a database, vector store, or file
   • Persists across conversations
   • Retrieved by the agent as needed
   • Perfect for: user preferences, past research, knowledge bases

3. EPISODIC MEMORY (experience)
   • Records of past tasks and their outcomes
   • Helps the agent avoid repeating mistakes
   • Can be used for few-shot examples in the prompt
   • Perfect for: improving over time, personalisation
""")

# Implement a simple external memory store
class AgentMemory:
    """Key-value memory that the agent can read and write."""

    def __init__(self):
        self._store = {}

    def remember(self, key: str, value: str) -> dict:
        """Store a fact for later retrieval."""
        self._store[key] = {"value": value, "stored_at": datetime.datetime.utcnow().isoformat()}
        return {"status": "stored", "key": key}

    def recall(self, key: str) -> dict:
        """Retrieve a stored fact by key."""
        if key in self._store:
            return {"key": key, **self._store[key]}
        # Fuzzy match
        matches = [k for k in self._store if key.lower() in k.lower()]
        if matches:
            return {"similar_keys": matches}
        return {"error": f"No memory found for '{key}'"}

    def list_memories(self) -> dict:
        """List all stored memory keys."""
        return {"keys": list(self._store.keys()), "count": len(self._store)}


memory = AgentMemory()

# Pre-populate with some user context
memory.remember("user_name", "Alex")
memory.remember("user_city", "London")
memory.remember("user_preference_units", "metric")

print("Memory store:", memory.list_memories())

In [ ]:
# Build memory tools
memory_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="remember",
            description="Store a fact or user preference in memory for later use.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "key":   types.Schema(type=types.Type.STRING, description="A descriptive key for the fact"),
                    "value": types.Schema(type=types.Type.STRING, description="The value or fact to store"),
                },
                required=["key", "value"],
            ),
        ),
        types.FunctionDeclaration(
            name="recall",
            description="Retrieve a stored fact from memory by key.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "key": types.Schema(type=types.Type.STRING, description="The memory key to retrieve"),
                },
                required=["key"],
            ),
        ),
        types.FunctionDeclaration(
            name="list_memories",
            description="List all stored memory keys to see what the agent knows.",
            parameters=types.Schema(type=types.Type.OBJECT, properties={}),
        ),
    ]
)

memory_registry = {
    "remember": memory.remember,
    "recall": memory.recall,
    "list_memories": memory.list_memories,
}


def run_agent(query, tools, registry, system_prompt="", max_steps=8, verbose=True):
    user_content = f"{system_prompt}\n\n{query}" if system_prompt else query
    contents = [types.Content(role="user", parts=[types.Part(text=user_content)])]
    for _ in range(max_steps):
        response = client.models.generate_content(
            model=MODEL, contents=contents,
            config=types.GenerateContentConfig(tools=tools),
        )
        part = response.candidates[0].content.parts[0]
        if not hasattr(part, "function_call") or part.function_call is None:
            return part.text.strip()
        fn_call = part.function_call
        fn_name, fn_args = fn_call.name, dict(fn_call.args)
        if verbose:
            print(f"  → {fn_name}({fn_args})")
        fn_result = registry.get(fn_name, lambda **k: {"error": "not found"})(**fn_args)
        contents.append(types.Content(role="model", parts=[types.Part(function_call=fn_call)]))
        contents.append(types.Content(role="user", parts=[types.Part(
            function_response=types.FunctionResponse(name=fn_name, response=fn_result)
        )]))
    return "Max steps reached."


# Test: agent uses memory to personalise its response
MEMORY_SYSTEM = """You are a personalised assistant. Before responding to questions, 
check your memory for relevant user context. Use recall() to look up what you know 
about the user. Store new important facts with remember()."""

print("Test 1: Recall user preferences")
answer = run_agent(
    "I'd like a weather comparison for my city vs Paris.",
    [memory_tool], memory_registry, system_prompt=MEMORY_SYSTEM
)
print(f"Answer: {answer}")

---

## 4. Planning Before Acting

In [ ]:
from pydantic import BaseModel

class Plan(BaseModel):
    steps: list[str]          # ordered action steps
    tools_needed: list[str]   # which tools will be required
    reasoning: str            # why this plan achieves the goal


def plan_task(task: str, available_tools: list[str]) -> Plan:
    """Ask Gemini to plan how to accomplish a task before acting."""
    response = client.models.generate_content(
        model=MODEL,
        contents=f"""You are a planning agent. Create a step-by-step plan to accomplish the task.
List the specific tools you'll call and the order of operations.

Available tools: {available_tools}

Task: {task}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Plan,
            temperature=0.0,
        ),
    )
    return Plan.model_validate_json(response.text)


# Plan a complex multi-step task before executing
complex_task = (
    "Which of the three cities — London, Paris, or Tokyo — has the highest population density? "
    "Show your calculations and give the answer in people per km²."
)

print("Generating plan...")
plan = plan_task(complex_task, ["lookup", "calculate"])

print(f"Plan ({len(plan.steps)} steps):")
for i, step in enumerate(plan.steps, 1):
    print(f"  {i}. {step}")
print(f"Tools needed: {plan.tools_needed}")
print(f"Reasoning: {plan.reasoning}")

print("\nExecuting plan...")
result = run_react_agent(complex_task)

---

## 5. When NOT to Use an Agent

Agents add complexity and latency — don't use them when simpler approaches work.

In [ ]:
print("""
DECISION GUIDE: Agent vs Simpler Approach

Use an AGENT when:
  ✅ Task requires multiple steps with data dependencies between them
  ✅ Task requires real-time or private data the LLM doesn't have
  ✅ Task has unpredictable structure — you can't pre-define all steps
  ✅ Task requires acting in the world (writing files, calling APIs)
  ✅ Error in one step should cause re-planning (self-correction)

  Examples:
    • "Research competitor pricing and summarise it in a table"
    • "Book a meeting based on all attendees' calendar availability"
    • "Debug this code by running it, reading the error, and fixing it"


Use a STANDARD LLM CALL when:
  ✅ Task is a single transformation (translate, summarise, classify)
  ✅ Task only needs information already in the LLM's training data
  ✅ Structure is completely predictable
  ✅ Speed/cost is critical

  Examples:
    • "Translate this paragraph to Spanish"
    • "Classify this support ticket into one of 5 categories"
    • "Write a commit message for this diff"


Use RAG (not an agent) when:
  ✅ Task needs information from a specific knowledge base
  ✅ The retrieval strategy is fixed (not dynamic)
  ✅ No need to take actions or iterate

  Examples:
    • "Answer questions about company policy"
    • "Find relevant documentation for this error message"
""")

---

## 🏋️ Exercise 1: Self-Reflective Agent

Build an agent that checks its own answer quality and retries if needed.

In [ ]:
# Exercise 1: Self-reflective agent
# Pattern:
#   1. Generate an answer
#   2. Score the answer (1-5) with a critic prompt
#   3. If score < 4, regenerate with the critique as context
#   4. Return after at most 3 attempts
#
# TODO:
# Implement critique_answer(question, answer, context) → (score, feedback)
# Implement self_reflective_agent(question, tools, registry) → answer
# Test on: "What is the population density of Tokyo?"
# The first answer might be vague — the critique should push for a precise number.

class Critique(BaseModel):
    score: int       # 1-5
    feedback: str    # Specific improvement suggestions
    is_acceptable: bool


def critique_answer(question: str, answer: str) -> Critique:
    # TODO: Implement — score the answer and provide feedback
    pass


def self_reflective_agent(question: str, tools, registry, max_attempts: int = 3) -> str:
    # TODO:
    # 1. Run run_react_agent() to get initial answer
    # 2. Critique the answer
    # 3. If not acceptable and attempts remaining, retry with critique in prompt
    # 4. Return the final answer
    pass


# TODO: Test here

---

## 🏋️ Exercise 2: Task Decomposition

For a complex multi-part question, have the agent break it into sub-tasks before solving.

In [ ]:
# Exercise 2: Sub-task decomposition
# TODO:
# 1. Write decompose_task(query) → list[str] that splits a complex question
#    into independent sub-questions
#
# 2. For each sub-question, run run_react_agent() independently
#
# 3. Combine sub-answers into a final comprehensive response
#
# Test on:
#   "Compare London and Tokyo: which has higher population density,
#    which has a lower area, and who is the current leader of each city?"

class SubTasks(BaseModel):
    sub_questions: list[str]


def decompose_task(query: str) -> list[str]:
    # TODO: Use Gemini to decompose the query into independent sub-questions
    pass


def decomposed_agent(query: str, tools, registry) -> str:
    # TODO: Decompose → solve each sub-task → combine
    pass


# TODO: Test here

---

## Key Takeaways

| Concept | Detail |
|---------|--------|
| Agent loop | Perceive → Reason → Act → repeat until done |
| ReAct | Adds explicit thought traces before each action |
| Planning agent | Decomposes task into ordered steps before acting |
| Self-reflection | Agent critiques and retries its own outputs |
| Memory types | In-context (conversation), external (database), episodic (past outcomes) |
| When to use | Multi-step, real-time data, dynamic structure, actions with effects |

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [ReAct Paper](https://arxiv.org/abs/2210.03629) | Paper | Original Reason + Act paper |
| [Building Effective Agents — Anthropic](https://www.anthropic.com/research/building-effective-agents) | Blog | Practical agent design guide |
| [Cognitive Architectures for LLM Agents](https://arxiv.org/abs/2309.02427) | Paper | Survey of agent memory and planning patterns |

---

**Next up:** [04_Building_a_ReAct_Agent.ipynb](./04_Building_a_ReAct_Agent.ipynb)